Language: **English (default)** | [Polski](lab.ipynb)

# Lab 05 -- recognizing overfitting and choosing regularization

Companion material to [`README.md`](README.md). Tabular classification
with informative and noise features (a different problem than the
lecture). You start by diagnosing a data leak in normalization, then
train a model that overfits, and experimentally compare regularization
methods.

In [ ]:
import copy  # noqa: F401 - used in section 5 (early stopping)

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data and split (given)

14 features (4 informative, 10 pure noise), 2 features with artificially
larger scale. 100 training, 200 validation, 200 test.

In [ ]:
def make_data(n, n_informative, n_noise, seed, noise_std=0.3):
    g = torch.Generator().manual_seed(seed)
    n_features = n_informative + n_noise
    X = torch.randn(n, n_features, generator=g)
    true_w = torch.randn(n_informative, generator=g)
    logits = X[:, :n_informative] @ true_w + torch.randn(n, generator=g) * noise_std
    y = (logits > 0).float()
    return X, y.unsqueeze(-1)


N_TRAIN, N_VAL, N_TEST = 100, 200, 200
X_all, y_all = make_data(N_TRAIN + N_VAL + N_TEST, n_informative=4, n_noise=10, seed=0)
X_all[:, 0] *= 15.0  # feature with a larger scale
X_all[:, 5] *= 8.0  # second feature with a larger scale

perm = torch.randperm(X_all.shape[0], generator=torch.Generator().manual_seed(1))
train_idx = perm[:N_TRAIN]
val_idx = perm[N_TRAIN : N_TRAIN + N_VAL]
test_idx = perm[N_TRAIN + N_VAL :]

X_train_raw, y_train = X_all[train_idx], y_all[train_idx]
X_val_raw, y_val = X_all[val_idx], y_all[val_idx]
X_test_raw, y_test = X_all[test_idx], y_all[test_idx]

print(f"N_train={len(train_idx)}, N_val={len(val_idx)}, N_test={len(test_idx)}")
print(
    f"std deviations (raw, before normalization), first 6 features: "
    f"{X_train_raw[:, :6].std(dim=0).tolist()}"
)

### Data leakage -- look before moving on

The code below is **deliberately buggy**. Read it carefully before
running the next cell.

In [ ]:
# WARNING: this code computes normalization statistics on all of X_all, BEFORE the split.
mean_leaky = X_all.mean(dim=0)
std_leaky = X_all.std(dim=0)

X_train_leaky = (X_train_raw - mean_leaky) / std_leaky
X_val_leaky = (X_val_raw - mean_leaky) / std_leaky
X_test_leaky = (X_test_raw - mean_leaky) / std_leaky

print(
    f"(buggy) mean of feature 0: {mean_leaky[0].item():.4f}  "
    f"std of feature 0: {std_leaky[0].item():.4f}"
)

**Task (Markdown, 3-5 sentences):** what exactly is the bug in the code
above? Why is this data leakage, even if the model "works well"? What
data, at what point, should not be visible when computing
`mean_leaky`/`std_leaky`?

_(Your answer here.)_

### The fix -- your task

Compute the normalization statistics **only** on `X_train_raw`, apply
them (without recomputing) to all three sets.

In [ ]:
def normalize_correctly(X_train_raw, X_val_raw, X_test_raw):
    """Returns (mean, std, X_train, X_val, X_test). Statistics ONLY from X_train_raw."""
    # TODO:
    # 1. compute mean, std EXCLUSIVELY from X_train_raw (dim=0)
    # 2. normalize X_train_raw, X_val_raw, X_test_raw with these same
    #    statistics (without recomputing separately for val/test!)
    # 3. return (mean, std, X_train, X_val, X_test)
    raise NotImplementedError("TODO: correct normalization")

In [ ]:
mean_correct, std_correct, X_train, X_val, X_test = normalize_correctly(
    X_train_raw, X_val_raw, X_test_raw
)
X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

print(
    f"(correct) mean of feature 0: {mean_correct[0].item():.4f}  "
    f"std of feature 0: {std_correct[0].item():.4f}"
)
print(
    f"difference of means (buggy - correct), feature 0: "
    f"{(mean_leaky[0] - mean_correct[0]).item():.4f}"
)
print(
    f"difference of std deviations (buggy - correct), feature 0: "
    f"{(std_leaky[0] - std_correct[0]).item():.4f}"
)

**Interpretation.** The differences in means/standard deviations for
feature 0 are clearly nonzero -- the buggy and correct normalization
statistics are **not** the same numbers, even though they come from the
same underlying dataset.

### Does the leak actually change the result? Let's check, instead of assuming (given)

A quick, small model trained on buggily and correctly normalized data --
compared on the corresponding (buggily/correctly normalized) test sets.
This is proof, not an assumption, which you will answer in question 2 of
section 7.

In [ ]:
def _quick_train_eval(X_tr, y_tr, X_eval, y_eval, seed):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(14, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(500):
        m.train()
        opt.zero_grad()
        loss = loss_fn(m(X_tr), y_tr)
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        acc = ((torch.sigmoid(m(X_eval)) > 0.5).float() == y_eval).float().mean().item()
    return acc


print("Test accuracy, several initialization seeds:")
for seed in [0, 1, 2]:
    acc_leaky = _quick_train_eval(X_train_leaky, y_train, X_test_leaky, y_test, seed)
    acc_correct = _quick_train_eval(X_train, y_train, X_test, y_test, seed)
    print(
        f"  seed={seed}: buggy normalization test_acc={acc_leaky:.3f}   "
        f"correct normalization test_acc={acc_correct:.3f}"
    )

## 2. Baseline model and an over-capacity model (given)

In [ ]:
def make_model(hidden_sizes, n_features=14, seed=0, dropout_p=0.0):
    torch.manual_seed(seed)
    layers = []
    in_dim = n_features
    for h in hidden_sizes:
        layers.append(nn.Linear(in_dim, h))
        layers.append(nn.ReLU())
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        in_dim = h
    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers).to(device)


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
        return (preds == y).float().mean().item()


majority_class = y_train.mean().round()
baseline_val_acc = (torch.full_like(y_val, majority_class.item()) == y_val).float().mean().item()
print(f"Baseline (majority class): val_acc={baseline_val_acc:.3f}")

MODEL_SEED = 1
HIDDEN = [32, 32]
print(
    f"Model [32,32] parameters: "
    f"{sum(p.numel() for p in make_model(HIDDEN, seed=MODEL_SEED).parameters())}"
)

## 3. Training loop -- your task

In [ ]:
def train_model(model, epochs, lr=0.001, weight_decay=0.0, use_adamw=False):
    """Returns (train_losses, val_losses). The whole X_train in one batch
    (100 examples) - no mini-batches needed here.
    """
    # TODO:
    # 1. build the optimizer: torch.optim.AdamW if use_adamw, otherwise
    #    torch.optim.Adam, with lr and weight_decay
    # 2. loss_fn = nn.BCEWithLogitsLoss()
    # 3. in a loop over epochs: model.train() -> zero_grad -> forward on the
    #    whole X_train -> loss -> backward() -> step() -> record the loss
    # 4. model.eval() -> under torch.no_grad() compute and record the loss on X_val/y_val
    raise NotImplementedError("TODO: train_model")

In [ ]:
baseline_model = make_model(HIDDEN, seed=MODEL_SEED)
baseline_tl, baseline_vl = train_model(baseline_model, epochs=1000, lr=0.001)

baseline_train_acc = accuracy(baseline_model, X_train, y_train)
baseline_val_acc_trained = accuracy(baseline_model, X_val, y_val)
print(
    f"Baseline [32,32] without regularization: train_acc={baseline_train_acc:.3f}  "
    f"val_acc={baseline_val_acc_trained:.3f}"
)
print(
    f"train_loss_final={baseline_tl[-1]:.4f}  "
    f"val_loss_min={min(baseline_vl):.4f}@{baseline_vl.index(min(baseline_vl))}  "
    f"val_loss_final={baseline_vl[-1]:.4f}"
)

plt.figure(figsize=(6, 4.5))
plt.plot(baseline_tl, label="train")
plt.plot(baseline_vl, label="validation")
plt.xlabel("epoch")
plt.ylabel("BCE")
plt.title("Baseline [32,32], no regularization")
plt.legend()
plt.tight_layout()
plt.show()

**Task (Markdown, 3-5 sentences):** what is the training vs. validation
accuracy? At which point of the curve do you see evidence of overfitting
-- not just "the losses are different", but the specific shape that shows
it?

_(Your answer here.)_

## 4. Weight decay and dropout -- your task

In [ ]:
# TODO:
# 1. for AdamW: test weight_decay from [0.0, 0.01, 0.1, 0.3] (use
#    train_model(..., weight_decay=wd, use_adamw=True)), save the results
#    INCLUDING THE TRAINED MODEL (e.g. in a dict
#    wd_results[wd] = (model, tl, vl, accuracy(model, X_val, y_val)))
#    - the model will be needed in section 6 for the final evaluation
# 2. for dropout: test p from [0.0, 0.2, 0.4, 0.6] (make_model(...,
#    dropout_p=p), train_model(...) without weight_decay), save analogously
#    in dropout_results
# 3. print for both: the parameter value, val_acc, val_loss_final
raise NotImplementedError("TODO: weight decay and dropout")

## 5. Early stopping -- your task

The loop's skeleton is given -- what is missing is the logic for
remembering and restoring the best state.

In [ ]:
def train_with_early_stopping(model, epochs, lr=0.001, patience=50, min_delta=1e-4):
    """Returns (model, train_losses, val_losses, best_epoch, stopped_epoch).
    The returned model MUST have the state from the best epoch restored.
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    best_val = float("inf")  # noqa: F841 - use in the TODO below
    best_state = None  # noqa: F841 - use in the TODO below
    best_epoch = -1
    epochs_since_improvement = 0  # noqa: F841 - use in the TODO below
    stopped_epoch = epochs
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        opt.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            v = loss_fn(model(X_val), y_val).item()
        val_losses.append(v)

        # TODO:
        # 1. if v < best_val - min_delta: update best_val, save a DEEP COPY
        #    of the model's state (copy.deepcopy(model.state_dict())) as
        #    best_state, set best_epoch = epoch, reset epochs_since_improvement
        # 2. otherwise: increment epochs_since_improvement; if it reached
        #    patience: set stopped_epoch = epoch and break out of the loop
        raise NotImplementedError("TODO: best-state/patience logic")

    # TODO: restore the best state into the model (model.load_state_dict(best_state))
    # (this code is unreachable until the TODO above stops raising - that is deliberate)
    return model, train_losses, val_losses, best_epoch, stopped_epoch

In [ ]:
es_model = make_model(HIDDEN, seed=MODEL_SEED)
es_model, es_tl, es_vl, es_best_epoch, es_stopped_epoch = train_with_early_stopping(
    es_model, epochs=1000, patience=50
)

print(f"Stopped at epoch {es_stopped_epoch}; best epoch = {es_best_epoch}")
print(f"Validation loss at the best epoch = {es_vl[es_best_epoch]:.4f}")
print(f"Validation loss at the last trained epoch = {es_vl[-1]:.4f}")

restored_val_loss = nn.BCEWithLogitsLoss()(es_model(X_val), y_val).item()
print(f"Validation loss AFTER restoring the model's state = {restored_val_loss:.4f}")
assert abs(restored_val_loss - es_vl[es_best_epoch]) < 1e-6, (
    "The model was not correctly restored to its best state!"
)
print("OK: restoring the best state works correctly.")
print(f"Validation accuracy after restoration: {accuracy(es_model, X_val, y_val):.3f}")

## 6. Controlled comparison and final evaluation -- your task

Choose **one** configuration with the best validation loss among all
those tested (sections 3-5). Train it (again, if needed) and evaluate it
once, on the test set.

In [ ]:
# TODO:
# 1. collect the validation loss of EVERY tested configuration together
#    with ITS TRAINED MODEL: baseline (baseline_model, baseline_vl[-1]),
#    every wd and dropout p from wd_results/dropout_results (which in
#    section 4 should already store the model), early stopping (es_model,
#    es_vl[es_best_epoch] - its best epoch, not the last one)
# 2. choose the configuration with the lowest validation loss - USE THE
#    ALREADY TRAINED MODEL for it (do not retrain from scratch - this way
#    you avoid accidentally changing the result through different randomness)
# 3. compute this model's loss and accuracy on X_test/y_test
#    (nn.BCEWithLogitsLoss(), accuracy(...)) - this is the only place in
#    the notebook where you use X_test
raise NotImplementedError("TODO: final evaluation")

## 7. Conclusions -- your task

Answer (Markdown, justified with real numbers):

1. Which regularization method gave the largest improvement over the
   baseline? Did it improve accuracy, loss, or both measures?
2. Did the normalization bug from section 1 have a large effect on the
   final result in your run, or a small one? Regardless of the answer --
   why is the procedure itself wrong, even if the effect turned out small?
3. What is at least one limitation of this experiment that means you
   should not generalize the conclusions to "method X is always best"?

_(Your answer here.)_